# 申鹤 RVC 语音 API
Colab 上跑，用 GPU 做中文申鹤声线转换。

**用法：Runtime → Run all，最后一格出 ngrok URL 即可。**

In [ ]:
# 1. 装依赖（约 3 分钟）
!pip install torch torchaudio fairseq -q 2>/dev/null
!pip install librosa soundfile pyworld numpy scipy -q 2>/dev/null
!pip install flask flask-cors -q 2>/dev/null
!pip install pyngrok huggingface_hub -q 2>/dev/null
!apt-get update -qq 2>/dev/null && apt-get install -y -qq ffmpeg 2>/dev/null

# 验证关键包
import torch, flask, librosa, soundfile
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}')
print(f'Flask {flask.__version__}')
print('✅ 依赖就绪')

In [ ]:
# 2. 克隆 RVC + 下载模型
import sys, os, subprocess

# 克隆 RVC
if not os.path.exists('/content/rvc'):
    !git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/rvc -q

RVC_ROOT = '/content/rvc'
sys.path.insert(0, RVC_ROOT)
sys.path.insert(0, f'{RVC_ROOT}/infer/lib')
sys.path.insert(0, f'{RVC_ROOT}/infer/modules')

# 下载 HuBERT
os.makedirs(f'{RVC_ROOT}/assets/hubert', exist_ok=True)
if not os.path.exists(f'{RVC_ROOT}/assets/hubert/hubert_base.pt'):
    from huggingface_hub import hf_hub_download
    print('下载 HuBERT...')
    hf_hub_download('lj1995/VoiceConversionWebUI', filename='hubert_base.pt',
                    local_dir=f'{RVC_ROOT}/assets/hubert')

# 下载申鹤模型
MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)
for f in ['genshin_impact/shenhe_e250_s11250.pth', 'genshin_impact/shenhe_pitch.index']:
    dst = os.path.join(MODEL_DIR, f)
    if not os.path.exists(dst):
        from huggingface_hub import hf_hub_download
        print(f'下载 {f}...')
        hf_hub_download('Cinnamomo/rvc_models', filename=f, local_dir=MODEL_DIR)

os.environ['weight_root'] = MODEL_DIR
os.environ['index_root'] = MODEL_DIR
print('✅ 模型就绪')

In [ ]:
# 3. 加载申鹤 RVC 模型
import torch
from infer.modules.vc.modules import VC

class Cfg:
    is_half = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_cpu = 2; x_pad = 3; x_query = 10; x_center = 60; x_max = 65
    f0method = 'harvest'

vc = VC(Cfg())
vc.get_vc('shenhe_e250_s11250.pth')
INDEX = os.path.join(MODEL_DIR, 'genshin_impact/shenhe_pitch.index')
print(f'✅ 申鹤模型加载完毕 | 设备: {Cfg.device}')

In [ ]:
# 4. 测试推理
import numpy as np, soundfile as sf
sf.write('/tmp/test.wav', np.zeros(16000, dtype=np.float32), 16000)
r = vc.vc_single(0, '/tmp/test.wav', 0, None, 'harvest', INDEX, '', 0.5, 3, 0, 0.25, 0.33)
print(f'✅ 推理通过: {r[0][:80]}')

In [ ]:
# 5. 启动 API 服务
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import asyncio, edge_tts, uuid, threading, time

app = Flask(__name__)
CORS(app)

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'device': str(Cfg.device)})

@app.route('/tts', methods=['POST'])
def tts():
    text = request.json.get('text', '')
    if not text:
        return jsonify({'error': 'text required'}), 400
    try:
        # 1. TTS
        mp3 = f'/tmp/tts_{uuid.uuid4().hex[:8]}.mp3'
        async def gen():
            await edge_tts.Communicate(text, 'zh-CN-XiaoxiaoNeural', rate='-10%').save(mp3)
        asyncio.run(gen())
        # 2. RVC
        r = vc.vc_single(0, mp3, 0, None, 'harvest', INDEX, '', 0.5, 3, 0, 0.25, 0.33)
        info, (sr, audio) = r
        if audio is None:
            return jsonify({'error': info}), 500
        # 3. 输出
        out = f'/tmp/shenhe_{uuid.uuid4().hex[:8]}.wav'
        arr = audio.cpu().numpy() if hasattr(audio, 'cpu') else audio
        import soundfile as sf
        sf.write(out, arr.reshape(-1), sr)
        return send_file(out, mimetype='audio/wav')
    except Exception as e:
        return jsonify({'error': str(e)}), 500

def start_ngrok():
    time.sleep(2)
    from pyngrok import ngrok
    ngrok.set_auth_token('2gJmme4OOxo6rMJBPaHTvZT4i5k_4aueJmUXEC62SLrWUsRHP')
    url = ngrok.connect(5000, 'http').public_url
    print(f'\n🔗 **公网 API: {url}**')
    print(f'   测试: curl {url}/health')
    print(f'\n📋 复制此 URL 发给开发者')

threading.Thread(target=start_ngrok).start()
print('启动 Flask...')
app.run(host='0.0.0.0', port=5000)